# Ariel 2025 — Notebook 2: ML Training (physics-based tabular models)

Train pipeline cho nhóm **machine learning trên feature vật lý** (không phải deep sequence):
calibrate → feature engineering → Target PCA → hồi quy theo họ mô hình → **PHC sigma calibration** → chọn model theo **Ariel GLL** → lưu weights (kèm calibrated sigma) → submission.

> Chạy bằng **CPU** (sklearn). GPU chỉ giúp LightGBM/XGBoost — bật `USE_GPU` nếu cần. Build feature là bước chậm nhất; có cache để chạy lại nhanh.

In [ ]:
# === Setup: clone running branch and make modules importable ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest →", CLONE_DIR)

CANDIDATE_SRC = [
    str(CLONE_DIR / "src"),
    "/kaggle/input/ariel-ml-src/src",
    "src", "../src",
]
for _p in CANDIDATE_SRC:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using src from:", _p); break
else:
    print("WARNING: src not found.")

# Optional GBM deps (LightGBM/XGBoost preinstalled on Kaggle; ngboost is not):
# !pip install -q ngboost

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = OUTPUT_DIR / "weights"; WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / "plots"; PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT exists:", DATA_ROOT.exists())


## 1. Cấu hình

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import ModelConfig
from dataset_builder import align_features_and_targets
from benchmark import benchmark_models
from training import cross_validate_model, train_model, search_n_components, build_gll_weighted_ensemble

# Phải KHỚP prepare_features.ipynb để load đúng file precomputed
LIMIT = None        # None = full
TIME_BINS = 128

# ---- training knobs ----
N_COMPONENTS = 24
N_SPLITS = 3                 # full data: 3 cho nhanh; tăng 5 nếu đủ thời gian
SIGMA_CAL_FRACTION = 0.2
USE_GPU = False              # chỉ tăng tốc lightgbm/xgboost
RANDOM_STATE = 42
MAX_PLANETS_BENCH = None     # thiếu RAM thì đặt 300-500

# ===== CHẠY NHIỀU LẦN: mỗi session chỉ chạy 1 nhóm họ model (để < 12h) =====
ALL_MODELS = ["ridge","lasso","elastic_net","bayesian_ridge","ard",
              "svr","kernel_ridge","knn",
              "random_forest","extra_trees","hist_gradient_boosting","xgboost","lightgbm","mlp"]
# Gợi ý chia nhóm (đặt MODELS_TO_RUN = 1 dòng mỗi lần chạy):
#   run 1 (nhẹ):  ["ridge","lasso","elastic_net","bayesian_ridge","ard","svr","kernel_ridge","knn","mlp"]
#   run 2 (cây):  ["random_forest","extra_trees","hist_gradient_boosting"]
#   run 3 (boost):["xgboost","lightgbm"]
#   run 4 (chậm): ["gaussian_process","ngboost"]  -> session riêng, đặt MAX_PLANETS_BENCH=300 & N_COMPONENTS=16
#                  (GPR là O(n^3) theo #planet; ngboost cần: !pip install ngboost)
MODELS_TO_RUN = ["ridge","lasso","elastic_net","bayesian_ridge","ard","svr","kernel_ridge","knn","mlp"]

# Cộng dồn benchmark qua các lần: trỏ tới benchmark.csv của lần trước (attach output cũ). None = lần đầu.
PREV_BENCHMARK = None         # vd: "/kaggle/input/ariel-prev/benchmark.csv"

RUN_PHC_ABLATION = False      # ablation PHC (có extra_trees -> hơi nặng)
MAKE_SUBMISSION = False       # bật ở SESSION CUỐI (build ensemble + submission)
print("config ok | models lần này:", MODELS_TO_RUN)


## 2. Load precomputed features
Features được build ở **`prepare_features.ipynb`** (CPU, chạy một lần) và push lên branch `running`. Notebook này chỉ load lại — `LIMIT`/`TIME_BINS` phải khớp.

In [ ]:
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
train_csv = PRECOMPUTED_DIR / f"features_train_L{LIMIT}_T{TIME_BINS}.csv"
test_csv = PRECOMPUTED_DIR / f"features_test_T{TIME_BINS}.csv"

def _resolve(path):
    for cand in [path, OUTPUT_DIR / path.name, Path("/kaggle/input/ariel-features") / path.name]:
        if Path(cand).exists():
            return Path(cand)
    return path

train_csv, test_csv = _resolve(train_csv), _resolve(test_csv)
assert train_csv.exists(), (
    f"Không thấy {train_csv.name}. Chạy prepare_features.ipynb (cùng LIMIT/TIME_BINS) rồi push lên branch 'running'."
)
print("Train features:", train_csv, "| test:", test_csv, "(", test_csv.exists(), ")")


In [ ]:
features = pd.read_csv(train_csv)
targets = pd.read_csv(DATA_ROOT / "train.csv")
X, y, groups, target_columns = align_features_and_targets(features, targets)
Xv = X.to_numpy(dtype=float)
print("X:", Xv.shape, "| y:", y.shape, "| planets:", len(np.unique(groups)),
      "| features:", Xv.shape[1], "| targets:", y.shape[1])


## 2b. Chẩn đoán: mean có học được không?
GLL chỉ cao khi **mean dự đoán hơn mean toàn cục**. Tính R² theo từng wavelength trên một split: R²>0 nghĩa là model giải thích được biến thiên của planet đó. Nếu R²≈0 ⇒ nút thắt là feature/signal (hoặc thiếu dữ liệu), không phải tầng calibration.

In [ ]:
from training import train_model

def per_wavelength_r2(name, n_components=N_COMPONENTS):
    res = train_model(Xv, y, model_name=name,
                      model_config=ModelConfig(n_components=n_components, random_state=RANDOM_STATE),
                      validation_fraction=0.25, groups=groups, random_state=RANDOM_STATE)
    yv = y[res.validation_index]; pv = res.prediction.mu
    ss_res = ((yv - pv) ** 2).sum(axis=0)
    ss_tot = ((yv - yv.mean(axis=0)) ** 2).sum(axis=0)
    return 1.0 - ss_res / np.maximum(ss_tot, 1e-12)

r2s = {}
for name in ["ridge", "extra_trees"]:
    r2 = per_wavelength_r2(name); r2s[name] = r2
    print(f"{name:12s} mean R²={r2.mean():+.3f} | median={np.median(r2):+.3f} | "
          f"% wavelength R²>0: {(r2 > 0).mean() * 100:.0f}%")

plt.figure(figsize=(9, 3))
for name, r2 in r2s.items():
    plt.plot(r2, label=name, lw=1)
plt.axhline(0, color="k", lw=0.6); plt.ylim(-1, 1)
plt.xlabel("wavelength index"); plt.ylabel("R² (val)"); plt.legend(); plt.title("Mean predictiveness per wavelength")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "diag_r2_per_wavelength.png", dpi=150, bbox_inches="tight"); plt.show()


## 3. So sánh các họ mô hình (benchmark, official GLL) — CHẠY NHIỀU LẦN
Mỗi session chỉ benchmark `MODELS_TO_RUN` (1 nhóm họ) để < 12h, rồi **cộng dồn** vào `benchmark.csv`.
Lần sau: attach output lần trước, đặt `PREV_BENCHMARK` trỏ tới `benchmark.csv` cũ → bảng tự gộp.
Cùng GroupKFold folds, cùng `n_components`. Model thiếu dep → `skipped`.

In [ ]:
Xb, yb, gb = Xv, y, groups
if MAX_PLANETS_BENCH is not None:
    keep = np.isin(groups, np.unique(groups)[:MAX_PLANETS_BENCH])
    Xb, yb, gb = Xv[keep], y[keep], groups[keep]
    print("Benchmark subset:", Xb.shape)

bench_cfg = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE,
                        sigma_per_target=True, use_gpu=USE_GPU)  # PHC per-wavelength: GLL công bằng
result = benchmark_models(Xb, yb, model_names=MODELS_TO_RUN, model_config=bench_cfg,
                          n_splits=N_SPLITS, groups=gb, random_state=RANDOM_STATE,
                          sigma_cal_fraction=SIGMA_CAL_FRACTION)
new_rows = result.to_frame()

# Cộng dồn với các lần chạy trước (model trùng tên -> lấy kết quả mới)
RESULTS_CSV = OUTPUT_DIR / "benchmark.csv"
prev = None
for cand in [PREV_BENCHMARK, RESULTS_CSV]:
    if cand is not None and Path(cand).exists():
        prev = pd.read_csv(cand); print("Nạp benchmark trước:", cand, prev.shape); break
table = (pd.concat([prev[~prev["model"].isin(new_rows["model"])], new_rows], ignore_index=True)
         if prev is not None else new_rows)
table = table.sort_values(["family", "ariel_gll_score"], ascending=[True, False]).reset_index(drop=True)
table.to_csv(RESULTS_CSV, index=False)

print("\n=== Benchmark TÍCH LŨY (official Ariel GLL) ===")
with pd.option_context("display.max_rows", None, "display.width", 200):
    print(table.to_string(index=False))
ok = table[table["status"] == "ok"]
if len(ok):
    b = ok.loc[ok["ariel_gll_score"].idxmax()]
    print("Best so far:", b["model"], "=", round(b["ariel_gll_score"], 4))
table


In [ ]:
ok = table[table["status"] == "ok"].sort_values("ariel_gll_score")
plt.figure(figsize=(8, max(3, 0.4 * len(ok))))
plt.barh(ok["model"], ok["ariel_gll_score"], color="steelblue")
plt.xlabel("Ariel GLL score (higher = better)"); plt.title("Model family comparison")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "benchmark_gll.png", dpi=150, bbox_inches="tight"); plt.show()


## 4. Chọn n_components (sweep nhanh & CHÍNH XÁC)
`search_n_components` fit **một lần** ở k lớn nhất rồi cắt component để chấm mọi k — kết quả **y hệt** grid đầy đủ nhưng nhanh hơn nhiều. Chỉ sweep model **rẻ** (linear); model nặng (RF/boosting/...) đã so sánh ở Section 3.

In [ ]:
from training import search_n_components

# Sweep n_components chỉ trên model rẻ; heavy models so sánh ở Section 3 (benchmark).
SEARCH_MODELS = ["bayesian_ridge", "ridge"]
N_COMPONENTS_GRID = [8, 12, 16, 24, 32, 48]

candidates = []
for name in SEARCH_MODELS:
    r = search_n_components(
        Xv, y, model_name=name, n_components_grid=N_COMPONENTS_GRID,
        base_config=ModelConfig(random_state=RANDOM_STATE, sigma_per_target=True, use_gpu=USE_GPU),
        n_splits=3, groups=groups, random_state=RANDOM_STATE,
        selection_metric="ariel_gll_score", sigma_cal_fraction=SIGMA_CAL_FRACTION,
    )
    candidates += r.candidates
bestc = max(candidates, key=lambda c: c.mean_metrics["ariel_gll_score"])
print("Best (n_components sweep):", bestc.model_name,
      "| n_components =", bestc.model_config.n_components,
      "| GLL =", round(bestc.mean_metrics["ariel_gll_score"], 4))


## 4b. Ablation PHC — calibration σ
So sánh 3 chế độ hiệu chỉnh sigma trên **cùng folds**: scalar (toàn cục) → per_wavelength (bước 1) → feature_conditioned (bước 2). Chỉ khác tầng calibration, cùng model & n_components → đo đóng góp của PHC theo Ariel GLL.

In [ ]:
if RUN_PHC_ABLATION:
    PHC_MODELS = ["bayesian_ridge", "extra_trees"]   # 1 linear + 1 tree đại diện
    phc_modes = {
        "scalar":              ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, use_gpu=USE_GPU),
        "per_wavelength":      ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, use_gpu=USE_GPU, sigma_per_target=True),
        "feature_conditioned": ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, use_gpu=USE_GPU, sigma_feature_conditioned=True),
    }
    rows = []
    for mname in PHC_MODELS:
        for mode, cfg in phc_modes.items():
            m = cross_validate_model(Xv, y, model_name=mname, model_config=cfg,
                                     n_splits=3, groups=groups, random_state=RANDOM_STATE,
                                     sigma_cal_fraction=SIGMA_CAL_FRACTION).mean_metrics
            rows.append({"model": mname, "calibration": mode, "ariel_gll_score": m["ariel_gll_score"],
                         "gaussian_nll": m["gaussian_nll"], "coverage_1sigma": m["coverage_1sigma"]})
    phc_table = pd.DataFrame(rows)
    phc_table.to_csv(OUTPUT_DIR / "phc_ablation.csv", index=False)
    print(phc_table.to_string(index=False))
else:
    phc_table = None
    print("RUN_PHC_ABLATION=False -> bỏ qua ablation PHC")


In [ ]:
if RUN_PHC_ABLATION and phc_table is not None:
    pivot = phc_table.pivot(index="model", columns="calibration", values="ariel_gll_score")
    pivot = pivot[["scalar", "per_wavelength", "feature_conditioned"]]
    pivot.plot(kind="bar", figsize=(8, 4))
    plt.ylabel("Ariel GLL score"); plt.title("Ablation PHC — sigma calibration"); plt.xticks(rotation=0)
    plt.legend(title="calibration"); plt.tight_layout()
    plt.savefig(PLOTS_DIR / "phc_ablation_gll.png", dpi=150, bbox_inches="tight"); plt.show()
    print(pivot)
else:
    print("(bỏ qua plot PHC)")


## 5. Train + lưu weights (kèm calibrated sigma)
Mỗi model fit bằng `train_model` trên train split và **calibrate σ trên holdout** (PHC per-wavelength) → artifact `.joblib` có σ đã hiệu chỉnh, dùng predict ngay.

In [ ]:
import joblib, gc

# Lưu weight cho NHÓM model lần này (MODELS_TO_RUN). Skip nếu file đã có (cộng dồn qua các lần).
save_cfg = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE,
                       sigma_per_target=True, use_gpu=USE_GPU)
VAL_FRACTION = 0.2

saved = []
for name in MODELS_TO_RUN:
    path = WEIGHTS_DIR / f"{name}.joblib"
    if path.exists():
        print("đã có, skip:", name); continue
    cfg = bestc.model_config if name == bestc.model_name else save_cfg
    try:
        res = train_model(Xv, y, model_name=name, model_config=cfg,
                          validation_fraction=VAL_FRACTION, groups=groups, random_state=RANDOM_STATE)
        joblib.dump({"model": res.model, "feature_columns": list(X.columns),
                     "target_columns": target_columns, "model_name": name,
                     "model_config": cfg, "val_metrics": res.evaluation.as_dict()}, path)
        saved.append(name)
        print(f"saved: {path}  (val GLL={res.evaluation.ariel_gll_score:.4f})")
    except Exception as exc:
        print("skip", name, "->", type(exc).__name__, exc)
    finally:
        gc.collect()
print("Đã lưu", len(saved), "model lần này vào", WEIGHTS_DIR)

# Ensemble chỉ build ở session cuối (cần fit lại các thành viên -> tốn thời gian)
if MAKE_SUBMISSION:
    ens = build_gll_weighted_ensemble(
        Xv, y, model_names=("bayesian_ridge", "random_forest", "kernel_ridge"),
        model_config=ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, sigma_per_target=True, use_gpu=USE_GPU),
        validation_fraction=VAL_FRACTION, groups=groups, random_state=RANDOM_STATE)
    joblib.dump({"ensemble": ens.ensemble, "model_names": ens.model_names, "weights": ens.weights,
                 "feature_columns": list(X.columns), "target_columns": target_columns},
                WEIGHTS_DIR / "gll_weighted_ensemble.joblib")
    print("ensemble weights:", dict(zip(ens.model_names, np.round(ens.weights, 3))))


## 6. Submission
Dùng best model (đã calibrate) để predict trên test features.

In [ ]:
from submission import infer_submission_schema, predict_submission, save_submission

if MAKE_SUBMISSION and test_csv.exists():
    bench = pd.read_csv(OUTPUT_DIR / "benchmark.csv")
    ok = bench[bench["status"] == "ok"].sort_values("ariel_gll_score", ascending=False)
    chosen = next((m for m in ok["model"] if (WEIGHTS_DIR / f"{m}.joblib").exists()), None)
    assert chosen is not None, "Chưa có weight cho model tốt nhất — train nó (MODELS_TO_RUN) trong session này."
    print("Submission dùng model:", chosen)
    art = joblib.load(WEIGHTS_DIR / f"{chosen}.joblib")
    test_features = pd.read_csv(test_csv)
    for col in art["feature_columns"]:
        if col not in test_features.columns:
            test_features[col] = 0.0
    schema = infer_submission_schema(sample_submission=pd.read_csv(DATA_ROOT / "sample_submission.csv"))
    submission = predict_submission(art["model"], test_features,
                                    feature_columns=art["feature_columns"], schema=schema)
    save_submission(submission, OUTPUT_DIR / "submission.csv")
    print("Saved submission:", submission.shape, "->", OUTPUT_DIR / "submission.csv")
else:
    print("MAKE_SUBMISSION=False hoặc thiếu test features -> bỏ qua submission.")


## Tổng kết
Output đã lưu trong `/kaggle/working`:
- Bảng: `benchmark.csv`, `phc_ablation.csv`
- Plot (`plots/`): `benchmark_gll.png`, `phc_ablation_gll.png`, `diag_r2_per_wavelength.png`
- Weights: `weights/*.joblib` (kèm calibrated σ + val_metrics), `weights/gll_weighted_ensemble.joblib`
- Submission: `submission.csv`
Dùng các bảng/plot này trực tiếp cho báo cáo.